In [27]:

!pip install -q flask flask-cors pyngrok gtts huggingface_hub
!pip install -q --upgrade youtube-transcript-api
!pip install -q "langchain==0.1.20" "pydantic<2.0"


import os
import json
import uuid
import base64
import io
import re
from datetime import datetime

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from gtts import gTTS
from huggingface_hub import InferenceClient
import youtube_transcript_api 

from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain.prompts import PromptTemplate
from kaggle_secrets import UserSecretsClient

print(" OK")


 OK


In [28]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HAGING FACE")
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
client = InferenceClient(token=os.environ["HF_TOKEN"], model="Qwen/Qwen2.5-VL-72B-Instruct")


In [29]:
def get_video_id(url: str):
   
    if not url:
        return None
    patterns = [
        r"(?:v=|/videos/|embed/|youtu\.be/|/v/|/shorts/)([0-9A-Za-z_-]{11})",
        r"^([0-9A-Za-z_-]{11})$",
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None


def fetch_transcript_text(video_id: str, lang_code: str, max_chars: int = 3000) -> str:
   
    from youtube_transcript_api import YouTubeTranscriptApi

    short_lang = (lang_code or "en").split("-")[0]

    
    try:
        ytt_api = YouTubeTranscriptApi()
        try:
            fetched = ytt_api.fetch(video_id, languages=[short_lang, "en"])
        except Exception:
            transcript_list = ytt_api.list(video_id)
            try:
                found = transcript_list.find_transcript([short_lang])
            except Exception:
                found = next(iter(transcript_list))
            fetched = found.fetch()
        text = " ".join(snippet.text for snippet in fetched)
        return text[:max_chars]
    except AttributeError:
        pass  

  
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=[short_lang, "en"])
    text = " ".join(t["text"] for t in transcript_list)
    return text[:max_chars]


def generate_audio_base64(text, lang_code):
    try:
        short_lang = lang_code.split('-')[0]
        tts = gTTS(text=text, lang=short_lang)

        
        fp = io.BytesIO()
        tts.write_to_fp(fp)
        fp.seek(0)

        return base64.b64encode(fp.read()).decode("utf-8")
    except Exception as e:
        print(f"Audio generation failed: {e}")
        return None


In [30]:

response_schemas = [
    ResponseSchema(
        name="flashcards",
        description=(
            "A JSON array of flashcard objects extracted from the transcript. "
            "Each object must have exactly these keys: "
            "term (the vocabulary word/phrase in the target language), "
            "meaning (its translation/meaning in the native language), "
            "pronunciation (a simple phonetic guide), "
            "example_sentence (an example sentence in the target language), "
            "example_translation (translation of that example sentence)."
        ),
    ),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

prompt_template = PromptTemplate(
    template="""You are an expert language teacher. Extract the {max_words} most important and useful vocabulary words or short phrases from the transcript below, to help a learner of the target language.

Target Language: {target_lang}
Native Language (for meanings): {native_lang}

Transcript:
{transcript}

Return ONLY the JSON object described below. No markdown code fences, no extra commentary.

{format_instructions}
""",
    input_variables=["target_lang", "native_lang", "max_words", "transcript"],
    partial_variables={"format_instructions": format_instructions},
)


In [31]:
def safe_parse_flashcards(raw_text: str):
   
    try:
        parsed = output_parser.parse(raw_text)
        cards = parsed.get("flashcards", [])
        if isinstance(cards, list):
            return cards
    except Exception:
        pass

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
            cards = parsed.get("flashcards", [])
            if isinstance(cards, list):
                return cards
        except Exception:
            pass
    return []


app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({"status": "healthy"}), 200


@app.route('/generate_flashcards', methods=['POST'])
def generate_flashcards_api():
    data = request.json or {}
    url = data.get('youtube_url')
    target_lang = data.get('target_language', 'English')
    native_lang = data.get('native_language', 'Arabic')
    max_words = data.get('max_words', 15)
    include_audio = data.get('include_audio', True)

    video_id = get_video_id(url)
    if not video_id:
        return jsonify({"error": "Invalid YouTube URL"}), 400

    try:
        
        try:
            transcript_text = fetch_transcript_text(video_id, target_lang)
            if not transcript_text.strip():
                return jsonify({"error": "there is no subtitle."}), 404
        except Exception as e:
            print(f"Transcript Error Details: {e}")
            return jsonify({"error": f"Failed to fetch transcript: {str(e)}"}), 400

       
        final_prompt = prompt_template.format(
            target_lang=target_lang,
            native_lang=native_lang,
            max_words=str(max_words),
            transcript=transcript_text,
        )

        try:
            
            completion = client.chat_completion(
                messages=[{"role": "user", "content": final_prompt}],
                max_tokens=1500,
                temperature=0.1,
            )
            llm_response = completion.choices[0].message.content
        except Exception as e:
            return jsonify({"error": f"LLM inference failed: {str(e)}"}), 502

       
        cards_data = safe_parse_flashcards(llm_response)

        flashcards = []
        for card in cards_data:
            term = card.get('term', '')
            if not term:
                continue
            audio_b64 = generate_audio_base64(term, target_lang) if include_audio else None

            flashcards.append({
                "id": str(uuid.uuid4()),
                "term": term,
                "meaning": card.get('meaning', ''),
                "pronunciation": card.get('pronunciation', ''),
                "example_sentence": card.get('example_sentence', ''),
                "example_translation": card.get('example_translation', ''),
                "audio_base64": audio_b64,
            })

        deck = {
            "video_id": video_id,
            "target_language": target_lang,
            "native_language": native_lang,
            "flashcards": flashcards[:max_words],
            "created_at": datetime.now().isoformat(),
        }

        return jsonify(deck), 200

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


In [ ]:

try:
    ngrok.kill()
except Exception:
    pass

public_url = ngrok.connect(5000)
print(f"=====================================================")
print(f" Copy the link :")
print(f" {public_url.public_url}")
print(f"=====================================================")

app.run(port=5000)


 Copy the link :
 https://showpiece-unsold-wife.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
